# アダプタ推論の検証 — HF transformers + PEFT でロードして生成する

`01_launch_training_job.ipynb` で学習した LoRA アダプタ（`model.tar.gz`）を、ベースモデルに載せて生成できるかを確認します。
検証は SageMaker Training Job（`ml.g5.2xlarge`、学習と同じイメージ）として `src/inference/infer_adapter.py` を実行し、結果を `output.tar.gz` の JSON で受け取ります。

確認する内容:

1. アダプタの中身（キーの内訳、`mtp.*` の LoRA 重みの有無）
2. HF の `Qwen3_5ForConditionalGeneration` に載せたとき、safetensors の各キーがモデルに入ったか（PEFT は未使用キーを黙って捨てるため自前で照合）
3. `AutoModelForCausalLM`（text-only クラス）に載せた場合の一致数
4. ベースモデル単体とアダプタ付きの生成結果の違い


In [ ]:
import os, json, tarfile, boto3, sagemaker
from sagemaker.pytorch import PyTorch

sess    = sagemaker.Session()
try:
    role = os.environ.get('SAGEMAKER_ROLE') or sagemaker.get_execution_role()
except Exception:
    raise SystemExit('ローカル実行時は SAGEMAKER_ROLE 環境変数に SageMaker 実行ロールの ARN を設定してください')
region  = boto3.Session().region_name
account = boto3.client('sts').get_caller_identity()['Account']
bucket  = sess.default_bucket()

IMAGE_REPO = 'nemo-automodel-sagemaker'
IMAGE_TAG  = '0.6.0-pt2.10-py313-cu130'
image_uri  = f'{account}.dkr.ecr.{region}.amazonaws.com/{IMAGE_REPO}:{IMAGE_TAG}'
print('sagemaker', sagemaker.__version__, '| region', region, '| bucket', bucket)
print('image_uri', image_uri)


## 1. 検証対象のアダプタ

学習ジョブ名を指定すると `model.tar.gz` の S3 URI を取得します。S3 URI を直接指定しても構いません。
プロンプトは学習時と同じ `validation` チャネル（`val.jsonl` の `prompt`）から先頭 N 件を使います。


In [ ]:
training_job_name = ''   # 例: 'automodel-qwen35-cooking-lora-2026-09-24-05-06-27-958'。空なら下の adapter_s3 を直接指定
adapter_s3 = ''          # 例: f's3://{bucket}/automodel-qwen35-cooking-lora-.../output/model.tar.gz'

if training_job_name:
    desc = sess.sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
    adapter_s3 = desc['ModelArtifacts']['S3ModelArtifacts']
    model_id   = json.loads(desc['HyperParameters'].get('model_id', '"Qwen/Qwen3.5-0.8B"'))
else:
    model_id = 'Qwen/Qwen3.5-0.8B'
assert adapter_s3, 'training_job_name か adapter_s3 を指定してください'

prefix = 'automodel/cooking_basics'
val_s3 = f's3://{bucket}/{prefix}/validation/'    # 01 の Notebook がアップロードした val.jsonl
print('adapter :', adapter_s3)
print('model_id:', model_id)
print('val     :', val_s3)


## 2. 検証ジョブの Estimator

`distribution` を指定しないので toolkit は `python infer_adapter.py --key value` で起動します（1 GPU）。
`src/inference/requirements.txt` の `peft` は toolkit が起動前にインストールします（学習用の `src/` とは分けてあります）。


In [ ]:
hyperparameters = {
    'model_id': model_id,
    'num_samples': 5,          # 生成するプロンプト数
    'max_new_tokens': 128,
    'also_causal_lm': 1,       # AutoModelForCausalLM 経路のキー一致も確認する
}

estimator = PyTorch(
    image_uri=image_uri,
    entry_point='infer_adapter.py',
    source_dir='../src/inference',
    role=role,
    base_job_name='automodel-verify-adapter',
    instance_type='ml.g5.2xlarge',
    instance_count=1,
    volume_size=50,
    max_run=3600,
    hyperparameters=hyperparameters,
    environment={
        'HF_HOME': '/tmp/hf',
        'HF_TOKEN': os.environ.get('HF_TOKEN', ''),
        'WANDB_MODE': 'disabled',
    },
)


## 3. 実行

`[verify]` で始まる行が検証結果です。`key check:` の `matched` / `unused` と `generations:` の行を確認します。


In [ ]:
estimator.fit({'adapter': adapter_s3, 'validation': val_s3}, wait=True, logs='All')
job_name = estimator.latest_training_job.name
print('job:', job_name)


## 4. 結果の取得

`output.tar.gz` の `adapter_inference.json` を取得して表示します。


In [ ]:
desc = sess.sagemaker_client.describe_training_job(TrainingJobName=job_name)
out_s3 = f"{desc['OutputDataConfig']['S3OutputPath'].rstrip('/')}/{job_name}/output/output.tar.gz"
print('status:', desc['TrainingJobStatus'], '| billable sec:', desc.get('BillableTimeInSeconds'))
print('output:', out_s3)

os.makedirs(f'artifacts/{job_name}', exist_ok=True)
sess.download_data(path=f'artifacts/{job_name}', bucket=out_s3.split('/')[2], key_prefix='/'.join(out_s3.split('/')[3:]))
with tarfile.open(f'artifacts/{job_name}/output.tar.gz') as t:
    t.extractall(f'artifacts/{job_name}')
res = json.load(open(f'artifacts/{job_name}/adapter_inference.json', encoding='utf-8'))

print('\n== adapter ==')
print(json.dumps({k: res['adapter'][k] for k in ('num_keys', 'groups')}, ensure_ascii=False, indent=2))
print('mtp keys:', len(res['adapter']['mtp_keys']))
print('\n== key check (ConditionalGeneration) ==')
print({k: v for k, v in res['key_check_conditional_generation'].items() if not k.endswith('_keys')})
print('unused (先頭 5):', res['key_check_conditional_generation']['unused_keys'][:5])
if 'key_check_causal_lm' in res:
    print('\n== key check (AutoModelForCausalLM) ==')
    print({k: v for k, v in res['key_check_causal_lm'].items() if not k.endswith('_keys')})
print('\n== generations ==')
for g in res['generations']:
    print('-' * 80)
    print('prompt  :', g['prompt'])
    print('expected:', g['expected'])
    print('base    :', g['base'])
    print('adapter :', g['adapter'])
    print('changed :', g['changed'])
